# 1. Import Libraries & Configuration

**Import Libraries**

In [1]:
import os
import time

import pandas as pd
import requests
from Bio import Entrez, Medline

try:
    import truststore
except ImportError:
    pass
else:
    truststore.inject_into_ssl()

**Configuration**

In [ ]:
EMAIL = "*"

# Europe PMC REST API
EUROPE_PMC_BASE_URL = "https://www.ebi.ac.uk/europepmc/webservices/rest/search"

# OpenAlex API
os.environ["OPENALEX_API_KEY"] = "*"

OPEN_ALEX_BASE_URL = "https://api.openalex.org/works"
OPEN_ALEX_API_KEY = os.environ["OPENALEX_API_KEY"]

# Crossref API
CROSSREF_BASE_URL = "https://api.crossref.org/works"
CROSSREF_HEADERS = {
    "User-Agent": f"IndonesiaFermentedFoodsMining/1.0 (mailto:{EMAIL})"
}

# Semantic Scholar API
os.environ["SEMANTIC_SCHOLAR_API_KEY"] = "*"

SEMANTIC_SCHOLAR_API_KEY = os.getenv("SEMANTIC_SCHOLAR_API_KEY")
SEMANTIC_SCHOLAR_BASE_URL = "https://api.semanticscholar.org/graph/v1/paper/search/bulk"
SEMANTIC_SCHOLAR_HEADERS = {"x-api-key": SEMANTIC_SCHOLAR_API_KEY} if SEMANTIC_SCHOLAR_API_KEY else {}

# PubMed
os.environ["NCBI_API_KEY"] = "*"

Entrez.email = EMAIL
Entrez.tool = "IndonesianFermentedFoodsMining"
Entrez.api_key = os.getenv("NCBI_API_KEY")

**Keyword Blocks**

In [3]:
FOOD_TERMS = [
    "tempe", "tempeh",
    "tape", "tapai",
    "oncom", "peuyeum",
    "tempoyak",
    "rusip",
    "budu",
    "bekasam", "bakasam", 
    "dadih", "dadiah"
]

COMPOUND_TERMS = [
    "polyamine*", "biogenic amine*", "putrescine",
    "spermidine", "spermine", "agmatine",
    "cadaverine", "histamine", "tyramine",
    "GABA", "gamma-aminobutyric acid",
    "amina biogenik", "poliamin",
    "asam gamma-aminobutirat"
]

MICROBE_GENE_TERMS = [
    "lactic acid bacteria", "Lactiplantibacillus",
    "Bacillus", "glutamate decarboxylase",
    "gadB", "ornithine decarboxylase",
    "agmatine deiminase", "metagenom*",
    "16S", "whole genome sequencing",
    "bakteri asam laktat"
]

REGION_TERMS = [
    "Indonesia*", "Java", "Sumatra",
    "Kalimantan", "Sulawesi", "Bali"
]

# 2. Europe PMC

## Query Rules

**1. Europe PMC API — Query Syntax**

**Endpoint:** `/rest/search` · **Parameter:** `query`

Unqualified searches cover indexed metadata and available full text.

| Operator | Syntax | Example | Description |
|---|---|---|---|
| AND | `AND` or space | `tempeh AND Indonesia` | Requires both terms |
| OR | `OR` | `tempeh OR tempe` | Requires at least one term |
| NOT | `NOT` | `fermentation NOT alcohol` | Excludes a term |
| Exact phrase | `"..."` | `"fermented food"` | Matches a phrase |
| Wildcard | `*` | `ferment*` | Matches a word prefix |
| Grouping | `( ... )` | `(tempe OR tempeh) AND Indonesia` | Groups expressions |
| Field search | `FIELD:term` | `TITLE_ABS:tempeh` | Restricts the search field |

**2. Query Rules**

1. Use uppercase `AND`, `OR`, and `NOT`; the default is `AND`.
2. Use quotes for phrases and parentheses for grouped concepts.
3. Use `TITLE:`, `ABSTRACT:`, or `TITLE_ABS:` to limit search scope.
4. Use `cursorMark` for pagination; request `resultType="core"` for abstracts.

**3. Example Queries**

```python
# Synonyms and region
query = "(tempe OR tempeh) AND Indonesia*"

# Title and abstract only
query = 'TITLE_ABS:("fermented food" AND Indonesia*)'

# Compounds
query = "(tempe OR tempeh) AND (polyamine* OR histamine OR GABA)"
```

[Query documentation](https://europepmc.org/help) · [REST API](https://europepmc.org/RestfulWebService)

## Mining

**1. Query**

In [4]:
europe_pmc_query = '("fermented food*" AND Indonesia*) OR "pangan fermentasi"'
print(europe_pmc_query)

("fermented food*" AND Indonesia*) OR "pangan fermentasi"


**2. Set parameters**

In [5]:
europe_pmc_params = {
    "query": europe_pmc_query,
    "format": "json",
    "resultType": "core",  # Request complete metadata, including abstractText.
    "pageSize": 100,
}

**3. Pagination**

In [ ]:
europe_pmc_papers = []
europe_pmc_params["cursorMark"] = "*"

while True:
    response = requests.get(
        EUROPE_PMC_BASE_URL, params=europe_pmc_params, timeout=30
    )
    response.raise_for_status()
    data = response.json()
    batch = data["resultList"]["result"]
    if not batch:
        break

    europe_pmc_papers.extend(batch)
    next_cursor = data.get("nextCursorMark")
    if not next_cursor or next_cursor == europe_pmc_params["cursorMark"]:
        break
    europe_pmc_params["cursorMark"] = next_cursor
    time.sleep(1.1)

print(f"Total Europe PMC papers: {len(europe_pmc_papers)}")

**4. Metadata extraction**

In [ ]:
europe_pmc_records = []

for paper in europe_pmc_papers:
    journal = (paper.get("journalInfo") or {}).get("journal") or {}
    pdf = next((
        item.get("url")
        for item in (paper.get("fullTextUrlList") or {}).get("fullTextUrl") or []
        if item.get("documentStyle") == "pdf"
    ), None)

    europe_pmc_records.append({
        "doi": paper.get("doi"),
        "pmid": paper.get("pmid"),
        "pmcid": paper.get("pmcid"),
        "title": paper.get("title"),
        "authors": paper.get("authorString"),
        "abstract": paper.get("abstractText"),
        "publication_year": int(paper["pubYear"]) if paper.get("pubYear") else None,
        "publication_date": paper.get("firstPublicationDate"),
        "journal": paper.get("journalTitle") or journal.get("title"),
        "is_open_access": {"Y": True, "N": False}.get(paper.get("isOpenAccess")),
        "openalex_id": None,
        "semantic_scholar_id": None,
        "semantic_scholar_corpus_id": None,
        "europe_pmc_id": paper.get("id"),
        "europe_pmc_source": paper.get("source"),
        "publication_type": "; ".join((paper.get("pubTypeList") or {}).get("pubType") or []) or None,
        "language": paper.get("language"),
        "citation_count": paper.get("citedByCount"),
        "url": f"https://europepmc.org/article/{paper['source']}/{paper['id']}" if paper.get("source") and paper.get("id") else None,
        "pdf_url": pdf,
        "query": europe_pmc_query,
        "source_database": "Europe PMC",
    })

europe_pmc_df = pd.DataFrame(europe_pmc_records, columns=[
    "doi", "pmid", "pmcid", "title", "authors", "abstract",
    "publication_year", "publication_date", "journal", "is_open_access",
    "openalex_id", "semantic_scholar_id", "semantic_scholar_corpus_id",
    "europe_pmc_id", "europe_pmc_source", "publication_type", "language",
    "citation_count", "url", "pdf_url", "source_database", "query",
], dtype=object)

**5. Display the first 10 rows**

In [ ]:
europe_pmc_df.head(10)

# 3. OpenAlex

## Query Rules

**1. OpenAlex API — Query Syntax**

**Endpoint:** `/works` · **Parameter:** `search`

Work searches cover titles, abstracts, full text, and keywords.

| Operator | Syntax | Example | Description |
|---|---|---|---|
| AND | `AND` or space | `tempeh AND Indonesia` | Requires both terms |
| OR | `OR` | `tempeh OR tempe` | Requires at least one term |
| NOT | `NOT` | `fermentation NOT alcohol` | Excludes a term |
| Phrase | `"..."` | `"fermented food"` | Matches adjacent terms |
| Grouping | `( ... )` | `(tempe OR tempeh) AND Indonesia` | Groups expressions |
| Wildcard | `*` or `?` | `ferment*` | Requires `search.exact` |
| Fuzzy search | `~N` | `ferment~1` | Allows character edits |
| Proximity | `"..."~N` | `"fermented food"~3` | Allows phrase slop |

**2. Query Rules**

1. Use uppercase Boolean operators; the default is `AND`.
2. `search` applies stemming and removes stop words.
3. Use `search.title_and_abstract` to limit scope to titles and abstracts.
4. Wildcards require `search.exact` and at least three preceding characters; leading wildcards are unsupported.
5. Use `cursor` for pagination; a valid API key is required beyond the first page.

**3. Example Queries**

```python
# Synonyms and region: use search
query = "(tempe OR tempeh) AND Indonesia"

# Phrase: use search
query = '"fermented food" AND Indonesia'

# Prefix matching: change the request parameter to search.exact
query = "(tempe OR tempeh) AND polyamine*"
```

[Query documentation](https://help.openalex.org/api/searching/) · [Pagination](https://help.openalex.org/api/paging/)

## Mining

**1. Query**

In [ ]:
openalex_query = '("fermented food" AND Indonesia) OR "pangan fermentasi"'
print(openalex_query)

**2. Set parameters**

In [ ]:
openalex_params = {
    "api_key": OPEN_ALEX_API_KEY,
    "search": openalex_query,
    "per_page": 100,
}

**3. Pagination**

In [ ]:
openalex_papers = []
openalex_params["cursor"] = "*"

while True:
    response = requests.get(
        OPEN_ALEX_BASE_URL, params=openalex_params, timeout=30
    )
    response.raise_for_status()
    data = response.json()
    batch = data["results"]
    if not batch:
        break

    openalex_papers.extend(batch)
    next_cursor = (data.get("meta") or {}).get("next_cursor")
    if not next_cursor or next_cursor == openalex_params["cursor"]:
        break
    openalex_params["cursor"] = next_cursor
    time.sleep(1.1)

print(f"Total OpenAlex papers: {len(openalex_papers)}")

**4. Metadata extraction**

In [ ]:
openalex_records = []

for paper in openalex_papers:
    ids = paper.get("ids") or {}
    location = paper.get("primary_location") or {}
    source = location.get("source") or {}
    oa = paper.get("open_access") or {}
    words = {
        position: word
        for word, positions in (paper.get("abstract_inverted_index") or {}).items()
        for position in positions
    }

    openalex_records.append({
        "doi": paper.get("doi"),
        "pmid": ids.get("pmid"),
        "pmcid": ids.get("pmcid"),
        "title": paper.get("title"),
        "authors": "; ".join(
            (author.get("author") or {}).get("display_name") or ""
            for author in paper.get("authorships") or []
        ) or None,
        "abstract": " ".join(words[position] for position in sorted(words)) or None,
        "publication_year": paper.get("publication_year"),
        "publication_date": paper.get("publication_date"),
        "journal": source.get("display_name"),
        "is_open_access": oa.get("is_oa"),
        "openalex_id": paper.get("id"),
        "semantic_scholar_id": None,
        "semantic_scholar_corpus_id": None,
        "europe_pmc_id": None,
        "europe_pmc_source": None,
        "publication_type": paper.get("type"),
        "language": paper.get("language"),
        "citation_count": paper.get("cited_by_count"),
        "url": location.get("landing_page_url"),
        "pdf_url": location.get("pdf_url"),
        "query": openalex_query,
        "source_database": "OpenAlex",
    })

openalex_df = pd.DataFrame(openalex_records, columns=[
    "doi", "pmid", "pmcid", "title", "authors", "abstract",
    "publication_year", "publication_date", "journal", "is_open_access",
    "openalex_id", "semantic_scholar_id", "semantic_scholar_corpus_id",
    "europe_pmc_id", "europe_pmc_source", "publication_type", "language",
    "citation_count", "url", "pdf_url", "source_database", "query",
], dtype=object)

**5. Display the first 10 rows**

In [ ]:
openalex_df.head(10)

# 4. Crossref

## Query Rules

**1. Crossref API — Query Syntax**

**Endpoint:** `/works` · **Parameter:** `query.bibliographic`

Searches bibliographic metadata and returns ranked matches.

| Operator / feature | Syntax | Example | Description |
|---|---|---|---|
| Free text | Plain text | `Indonesian fermented foods` | Finds ranked matches |
| AND / OR / NOT | Not supported | — | No Boolean query expressions |
| Phrase / grouping | No Boolean-style guarantee | — | Do not assume strict phrase or group matching |
| Field query | `query.author` | `query.author=Smith` | Searches a specific metadata field |
| Structured filter | `filter=field:value` | `filter=type:journal-article` | Restricts results |

**2. Query Rules**

1. Use bibliographic text rather than Boolean expressions.
2. Do not assume every query word must occur in each result.
3. Combine structured restrictions in `filter`, separated by commas.
4. Use `cursor` for pagination; stop when a page contains fewer items than `rows`.

**3. Example Queries**

```python
# General topic
query = "Indonesian fermented foods"

# Specific food
query = "tempeh Indonesia"

# Compound-focused topic
query = "tempeh polyamines"
```

[Query documentation](https://github.com/CrossRef/rest-api-doc#queries) · [Boolean support](https://community.crossref.org/t/advanced-set-boolean-operators/4596)

## Mining

**1. Query**

In [ ]:
crossref_query = "Indonesian fermented foods"
print(crossref_query)

**2. Set parameters**

In [ ]:
crossref_params = {
    "query.bibliographic": crossref_query,
    "filter": "type:journal-article",
    "rows": 100,
    "mailto": EMAIL,
}

**3. Pagination**

In [ ]:
crossref_papers = []
crossref_params["cursor"] = "*"

while True:
    response = requests.get(
        CROSSREF_BASE_URL,
        params=crossref_params,
        headers=CROSSREF_HEADERS,
        timeout=30,
    )
    response.raise_for_status()
    message = response.json()["message"]
    batch = message["items"]
    if not batch:
        break

    crossref_papers.extend(batch)
    if len(batch) < crossref_params["rows"]:
        break
    next_cursor = message.get("next-cursor")
    if not next_cursor or next_cursor == crossref_params["cursor"]:
        break
    crossref_params["cursor"] = next_cursor
    time.sleep(1.1)

print(f"Total Crossref papers: {len(crossref_papers)}")

**4. Metadata extraction**

In [ ]:
crossref_records = []

for paper in crossref_papers:
    date_parts = ((paper.get("published") or {}).get("date-parts") or [[]])[0]
    titles = paper.get("title") or []
    journals = paper.get("container-title") or []
    authors = [
        " ".join(filter(None, [author.get("given"), author.get("family")]))
        or author.get("name") or ""
        for author in paper.get("author") or []
    ]
    pdf = next((
        link.get("URL") for link in paper.get("link") or []
        if link.get("content-type") == "application/pdf"
    ), None)

    crossref_records.append({
        "doi": paper.get("DOI"),
        "pmid": None,
        "pmcid": None,
        "title": titles[0] if titles else None,
        "authors": "; ".join(authors) or None,
        "abstract": paper.get("abstract"),
        "publication_year": date_parts[0] if date_parts else None,
        "publication_date": "-".join(f"{part:02d}" for part in date_parts) or None,
        "journal": journals[0] if journals else None,
        "is_open_access": None,
        "openalex_id": None,
        "semantic_scholar_id": None,
        "semantic_scholar_corpus_id": None,
        "europe_pmc_id": None,
        "europe_pmc_source": None,
        "publication_type": paper.get("type"),
        "language": paper.get("language"),
        "citation_count": paper.get("is-referenced-by-count"),
        "url": paper.get("URL"),
        "pdf_url": pdf,
        "query": crossref_query,
        "source_database": "Crossref",
    })

crossref_df = pd.DataFrame(crossref_records, columns=[
    "doi", "pmid", "pmcid", "title", "authors", "abstract",
    "publication_year", "publication_date", "journal", "is_open_access",
    "openalex_id", "semantic_scholar_id", "semantic_scholar_corpus_id",
    "europe_pmc_id", "europe_pmc_source", "publication_type", "language",
    "citation_count", "url", "pdf_url", "source_database", "query",
], dtype=object)

**5. Display the first 10 rows**

In [ ]:
crossref_df.head(10)

# 5. Semantic Scholar

## Query Rules

**1. Semantic Scholar API — Query Syntax**

**Endpoint:** `/graph/v1/paper/search/bulk`

Searches are performed against publication titles and abstracts.

| Operator | Syntax | Example | Description |
|---|---|---|---|
| AND | `+` or space | `tempeh + Indonesia` | Both terms must appear |
| OR | `\|` | `tempeh \| tempe` | At least one term must appear |
| NOT | `-` | `fermentation -alcohol` | Excludes a term |
| Exact phrase | `"..."` | `"fermented food"` | Matches a phrase |
| Wildcard | `*` | `ferment*` | Matches words sharing a prefix |
| Grouping | `( ... )` | `(tempeh \| tempe) + Indonesia` | Groups logical expressions |
| Fuzzy search | `~N` | `ferment~2` | Matches terms within N character edits |
| Proximity | `"..."~N` | `"fermented food"~3` | Allows phrase proximity/slop |

**2. Query Rules**

1. Boolean and advanced operators are supported by `/paper/search/bulk`, NOT `/paper/search`.
2. AND is the default operator between terms.
3. Use `|` to combine synonyms.
4. Use `+` or spaces to require multiple concepts.
5. Use parentheses to control logical precedence.
6. Use quotation marks for multiword phrases.
7. Use `*` only for prefix matching.
8. Search is case-insensitive and applies English stemming.
9. Queries match titles and abstracts, not full-text content.
10. Use the `token` parameter for pagination.

**3. Example Queries**

```python
# Basic search
query = "tempeh Indonesia"

# Synonyms
query = "(tempeh | tempe) + Indonesia"

# Exact phrase
query = '"fermented food" + Indonesia'

# Multiple concepts
query = "(tempeh | tempe) + (fermentation | microorganism*)"

# Compounds
query = "(tempeh | tempe) + (polyamine* | histamine | GABA)"

# Microorganisms
query = "(tempeh | tempe) + (Rhizopus | Bacillus)"

# Exclusion
query = "fermentation + Indonesia -biofuel"
```

[Official documentation](https://api.semanticscholar.org/api-docs/graph#tag/Paper-Data/operation/get_graph_get_paper_bulk_search)

## Mining

**1. Query**

In [ ]:
semantic_scholar_query = '"fermented foods" + Indonesia'
print(semantic_scholar_query)

**2. Set parameters**

In [ ]:
semantic_scholar_params = {
    "query": semantic_scholar_query,
    "fields": (
        "paperId,corpusId,title,abstract,authors,year,publicationDate,"
        "journal,venue,publicationTypes,externalIds,url,isOpenAccess,"
        "openAccessPdf,citationCount"
    ),
}

**3. Pagination**

In [ ]:
semantic_scholar_papers = []
semantic_scholar_params.pop("token", None)

while True:
    response = requests.get(
        SEMANTIC_SCHOLAR_BASE_URL,
        headers=SEMANTIC_SCHOLAR_HEADERS,
        params=semantic_scholar_params,
        timeout=30,
    )
    response.raise_for_status()
    data = response.json()
    batch = data["data"]
    if not batch:
        break

    semantic_scholar_papers.extend(batch)
    token = data.get("token")
    if not token or token == semantic_scholar_params.get("token"):
        break
    semantic_scholar_params["token"] = token
    time.sleep(1.1)

print(f"Total Semantic Scholar papers: {len(semantic_scholar_papers)}")

**4. Metadata extraction**

In [ ]:
semantic_scholar_records = []

for paper in semantic_scholar_papers:
    external_ids = paper.get("externalIds") or {}
    pdf = paper.get("openAccessPdf") or {}
    journal = paper.get("journal") or {}

    semantic_scholar_records.append({
        "doi": external_ids.get("DOI"),
        "pmid": external_ids.get("PubMed"),
        "pmcid": external_ids.get("PubMedCentral"),
        "title": paper.get("title"),
        "authors": "; ".join(author.get("name") or "" for author in paper.get("authors") or []) or None,
        "abstract": paper.get("abstract"),
        "publication_year": paper.get("year"),
        "publication_date": paper.get("publicationDate"),
        "journal": journal.get("name") or paper.get("venue"),
        "is_open_access": paper.get("isOpenAccess"),
        "openalex_id": None,
        "semantic_scholar_id": paper.get("paperId"),
        "semantic_scholar_corpus_id": paper.get("corpusId"),
        "europe_pmc_id": None,
        "europe_pmc_source": None,
        "publication_type": "; ".join(paper.get("publicationTypes") or []) or None,
        "language": None,
        "citation_count": paper.get("citationCount"),
        "url": paper.get("url"),
        "pdf_url": pdf.get("url"),
        "query": semantic_scholar_query,
        "source_database": "Semantic Scholar",
    })

semantic_scholar_df = pd.DataFrame(semantic_scholar_records, columns=[
    "doi", "pmid", "pmcid", "title", "authors", "abstract",
    "publication_year", "publication_date", "journal", "is_open_access",
    "openalex_id", "semantic_scholar_id", "semantic_scholar_corpus_id",
    "europe_pmc_id", "europe_pmc_source", "publication_type", "language",
    "citation_count", "url", "pdf_url", "source_database", "query",
], dtype=object)

**5. Display the first 10 rows**

In [ ]:
semantic_scholar_df.head(10)

# 6. PubMed

## Query Rules

**1. PubMed / Biopython — Query Syntax**

**Endpoints:** `esearch.fcgi` and `efetch.fcgi` · **Search parameter:** `term`

Biopython sends PubMed queries through `Bio.Entrez`.

| Operator | Syntax | Example | Description |
|---|---|---|---|
| AND | `AND` | `tempeh AND Indonesia` | Requires both concepts |
| OR | `OR` | `tempeh OR tempe` | Requires either concept |
| NOT | `NOT` | `fermentation NOT alcohol` | Excludes a concept |
| Phrase | `"..."` | `"fermented food"` | Requests a phrase search |
| Wildcard | `*` | `ferment*` | Replaces zero or more characters |
| Grouping | `( ... )` | `(tempe OR tempeh) AND Indonesia` | Groups concepts |
| Field tag | `[field]` | `tempeh[tiab]` | Restricts the search field |

**2. Query Rules**

1. Use uppercase Boolean operators and parentheses; PubMed evaluates left to right.
2. `[tiab]` restricts titles/abstracts; `[mh]` searches MeSH terms.
3. Wildcards require at least four preceding characters and disable Automatic Term Mapping.
4. Quoted phrases depend on PubMed's phrase index; inspect search warnings for unmatched phrases.
5. Use ESearch history and EFetch batches. Above 10,000 matches, narrow or split the query.

**3. Example Queries**

```python
# Food synonyms and region
query = "(tempe[tiab] OR tempeh[tiab]) AND Indonesia[tiab]"

# Phrase and MeSH region
query = '"fermented food"[tiab] AND "Indonesia"[mh]'

# Compounds
query = "tempeh[tiab] AND (polyamine*[tiab] OR histamine[tiab] OR GABA[tiab])"
```

Install `biopython` in the notebook environment and set `EMAIL` to your contact email.
An optional NCBI API key is read from `NCBI_API_KEY`.
MEDLINE dates retain source text, such as `2024 Jan`; OA status is unavailable and set to `None`.

[PubMed search help](https://pubmed.ncbi.nlm.nih.gov/help/) · [Biopython Entrez](https://biopython.org/docs/latest/Tutorial/chapter_entrez.html) · [NCBI E-utilities](https://www.ncbi.nlm.nih.gov/books/NBK25499/)

## Mining

**1. Query**

In [ ]:
pubmed_query = '("fermented food*"[tiab] AND Indonesia*[tiab]) OR "pangan fermentasi"[tiab]'
print(pubmed_query)

**2. Set parameters**

In [ ]:
pubmed_params = {
    "db": "pubmed",
    "term": pubmed_query,
    "usehistory": "y",
    "retmax": 0,
}
pubmed_batch_size = 100

**3. Pagination**

In [ ]:
pubmed_papers = []

with Entrez.esearch(**pubmed_params) as handle:
    pubmed_search = Entrez.read(handle)

pubmed_count = int(pubmed_search["Count"])
if pubmed_count > 10000:
    raise ValueError("PubMed ESearch is limited to 10,000 matches. Narrow the query or split it by date.")

for start in range(0, pubmed_count, pubmed_batch_size):
    with Entrez.efetch(
        db="pubmed",
        query_key=pubmed_search["QueryKey"],
        WebEnv=pubmed_search["WebEnv"],
        retstart=start,
        retmax=pubmed_batch_size,
        rettype="medline",
        retmode="text",
    ) as handle:
        pubmed_papers.extend(Medline.parse(handle))

print(f"Total PubMed papers: {len(pubmed_papers)}")

**4. Metadata extraction**

In [ ]:
pubmed_records = []

for paper in pubmed_papers:
    doi = next((
        identifier.removesuffix(" [doi]")
        for identifier in (paper.get("AID") or []) + (paper.get("LID") or [])
        if identifier.endswith(" [doi]")
    ), None)
    publication_date = paper.get("DP")

    pubmed_records.append({
        "doi": doi,
        "pmid": paper.get("PMID"),
        "pmcid": paper.get("PMC"),
        "title": paper.get("TI"),
        "authors": "; ".join(paper.get("FAU") or paper.get("AU") or []) or None,
        "abstract": paper.get("AB"),
        "publication_year": int(publication_date[:4]) if publication_date and publication_date[:4].isdigit() else None,
        "publication_date": publication_date,
        "journal": paper.get("JT"),
        "is_open_access": None,
        "openalex_id": None,
        "semantic_scholar_id": None,
        "semantic_scholar_corpus_id": None,
        "europe_pmc_id": None,
        "europe_pmc_source": None,
        "publication_type": "; ".join(paper.get("PT") or []) or None,
        "language": "; ".join(paper.get("LA") or []) or None,
        "citation_count": None,
        "url": f"https://pubmed.ncbi.nlm.nih.gov/{paper['PMID']}/" if paper.get("PMID") else None,
        "pdf_url": None,
        "query": pubmed_query,
        "source_database": "PubMed",
    })

pubmed_df = pd.DataFrame(pubmed_records, columns=[
    "doi", "pmid", "pmcid", "title", "authors", "abstract",
    "publication_year", "publication_date", "journal", "is_open_access",
    "openalex_id", "semantic_scholar_id", "semantic_scholar_corpus_id",
    "europe_pmc_id", "europe_pmc_source", "publication_type", "language",
    "citation_count", "url", "pdf_url", "source_database", "query",
], dtype=object)

**5. Display the first 10 rows**

In [ ]:
pubmed_df.head(10)